In [ ]:

# Standard Library
import os
import re
import json
import time
import uuid
import sqlite3
import hashlib
import logging
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional,cast
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
import pickle
# Environment Variables
from dotenv import load_dotenv

# Numerical Computing
import numpy as np

# Data Processing
import pandas as pd

# NLP
from prompt_toolkit import prompt
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector Search
import faiss

# Knowledge Graph
import networkx as nx

# LLM Client (OpenAI SDK)
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam


# Retry Logic
from tenacity import retry, stop_after_attempt, wait_exponential

# Progress Bars
from tqdm import tqdm


from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
import os
from dotenv import load_dotenv
from datetime import datetime, UTC

datetime.now(UTC).isoformat()

# CONFIGURATION

load_dotenv()

@dataclass
class Config:
    NVIDIA_API_KEY: str = os.getenv("NVIDIA_API_KEY", "")
    BASE_URL: str = "https://integrate.api.nvidia.com/v1"

    LLM_MODEL: str = "nvidia/nemotron-3-ultra-550b-a55b"
    EMBEDDING_MODEL: str ="sentence-transformers/all-MiniLM-L6-v2"
    SPACY_MODEL: str = "en_core_web_sm"

    DATABASE_PATH: Path = Path("personal_memory.db")
    FAISS_INDEX_PATH: Path = Path("memory_index.faiss")
    GRAPH_PATH: Path = Path("memory_graph.pkl")
    LOG_PATH: Path = Path("memory.log")

    EMBEDDING_DIMENSION: int = 384

    MAX_NOTE_LENGTH: int = 10000
    MIN_NOTE_LENGTH: int = 20

    SUMMARY_MAX_WORDS: int = 120
    MAX_KEYWORDS: int = 15
    MAX_ENTITIES: int = 50
    MAX_RELATIONSHIPS: int = 50

    IMPORTANCE_THRESHOLD: float = 0.50
    DUPLICATE_THRESHOLD: float = 0.95
    SEARCH_SIMILARITY_THRESHOLD: float = 0.60

    TOP_K_VECTOR_RESULTS: int = 10
    TOP_K_KEYWORD_RESULTS: int = 10
    TOP_K_GRAPH_RESULTS: int = 10

    TEMPERATURE: float = 0.2
    MAX_TOKENS: int = 8192
    RETRY_ATTEMPTS: int = 3
    REQUEST_TIMEOUT: int = 60

    PROMPTS: Dict[str, str] = field(default_factory=lambda: {
        "summary": "Summarize the note while preserving all important facts.",
        "memory_extraction": "Extract structured memories as valid JSON.",
        "consolidation": "Merge related memories into higher-level memories.",
        "analysis": "Analyze memories for goals, skills, beliefs, interests, habits, personality, contradictions and predictions.",
        "qa": "Answer only using the retrieved memories. If unsupported, explicitly say so."
    })


config = Config()


def load_config():
    return config


def get_prompt(name: str):
    return config.PROMPTS[name]


#  DATA LAYER


DB = str(config.DATABASE_PATH)


def get_connection():
    conn = sqlite3.connect(DB)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database():
    tables = {
        "notes": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            uuid TEXT UNIQUE,
            raw_text TEXT,
            clean_text TEXT,
            summary TEXT,
            importance REAL,
            created_at TEXT
        """,
        "memories": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            memory_type TEXT,
            content TEXT,
            confidence REAL,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "topics": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            topic TEXT,
            score REAL,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "entities": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            entity TEXT,
            label TEXT,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "relationships": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_entity TEXT,
            relationship TEXT,
            target_entity TEXT,
            confidence REAL
        """,
        "embeddings": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            vector BLOB,
            dimension INTEGER,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "observations": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            description TEXT,
            confidence REAL
        """,
        "predictions": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            prediction TEXT,
            probability REAL,
            created_at TEXT
        """,
        "metadata": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            key TEXT,
            value TEXT,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """
    }

    conn = get_connection()
    cur = conn.cursor()

    for table, schema in tables.items():
        cur.execute(f"CREATE TABLE IF NOT EXISTS {table} ({schema})")

    conn.commit()
    conn.close()

def save_record(table: str, data: dict):

    conn = get_connection()
    cur = conn.cursor()

    cleaned = {}

    for k, v in data.items():

        if isinstance(v, (dict, list)):
            cleaned[k] = json.dumps(v)

        else:
            cleaned[k] = v

    cols = ",".join(cleaned.keys())
    vals = ",".join("?" * len(cleaned))

    cur.execute(
        f"INSERT INTO {table} ({cols}) VALUES ({vals})",
        tuple(cleaned.values())
    )

    conn.commit()

    rowid = cur.lastrowid

    conn.close()

    return rowid


def load_records(table: str, where: str = "", params=()):
    conn = get_connection()
    cur = conn.cursor()

    query = f"SELECT * FROM {table}"
    if where:
        query += " WHERE " + where

    cur.execute(query, params)

    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows


def update_record(table: str, data: dict, where: str, params=()):
    conn = get_connection()
    cur = conn.cursor()

    fields = ",".join(f"{k}=?" for k in data.keys())

    cur.execute(
        f"UPDATE {table} SET {fields} WHERE {where}",
        tuple(data.values()) + tuple(params)
    )

    conn.commit()
    conn.close()


def delete_record(table: str, where: str, params=()):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute(
        f"DELETE FROM {table} WHERE {where}",
        params
    )

    conn.commit()
    conn.close()


def search_notes(keyword: str):
    return load_records(
        "notes",
        "clean_text LIKE ? OR summary LIKE ?",
        (f"%{keyword}%", f"%{keyword}%")
    )


#  LLM CLIENT


if not config.NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found in .env")

client = OpenAI(
    base_url=config.BASE_URL,
    api_key=config.NVIDIA_API_KEY,
)


@retry(
    stop=stop_after_attempt(config.RETRY_ATTEMPTS),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=True,
)
def call_llm(prompt: str, schema: dict | None = None) -> Any:
    """
    Central entry point for every LLM request.
    """

    system_prompt = (
        "You are a structured memory extraction assistant. "
        "Return only valid JSON when requested."
        "Return exactly one JSON object."
        "Rules:"
        " - Output ONLY JSON."
        " - No markdown."
        " - No ``json."
        " - No explanations."
    " -  No comments."
    " - Every key must exist."
    " - Arrays must always exist even if empty."
    " - The response must be parseable by Python json.loads()."
    )

    messages = cast(
    list[ChatCompletionMessageParam],
    [
        {
            "role": "system",
            "content": "You are a structured memory extraction assistant.",
        },
        {
            "role": "user",
            "content": prompt,
        },
    ],
)


    response = client.chat.completions.create(
        model=config.LLM_MODEL,
        messages=messages,
        temperature=config.TEMPERATURE,
        max_tokens=config.MAX_TOKENS,
        timeout=config.REQUEST_TIMEOUT,
    )

    print("LLM returned successfully")

    if not response.choices:
        raise ValueError("LLM returned no choices.")

    text = response.choices[0].message.content

    if text is None:
        raise ValueError("LLM returned empty content.")

    text = text.strip()

    # Remove Markdown code fences if present
    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    if schema is None:
        return text

    try:
        return json.loads(text)

    except json.JSONDecodeError as e:
        raise ValueError(
            f"Invalid JSON returned by LLM:\n{text}"
        ) from e


# MODULE 4 : NOTES PROCESSING


nlp = spacy.load(
    config.SPACY_MODEL,
    disable=["parser", "lemmatizer", "textcat"]
)
embedding_model = SentenceTransformer(config.EMBEDDING_MODEL)
tfidf = TfidfVectorizer(stop_words="english")

@dataclass
class ProcessedNote:
    raw_text: str
    clean_text: str
    summary: str
    embedding: np.ndarray
    importance: float
    keywords: List[str]
    metadata: Dict[str, Any]

def clean_text(text: str) -> str:
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def normalize_text(text: str) -> str:
    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

def extract_keywords(text: str) -> List[str]:

    if len(text.split()) < 3:
        return []

    try:

        vectorizer = tfidf

        matrix = vectorizer.fit_transform([text])

        names = vectorizer.get_feature_names_out()

        scores = np.asarray(matrix.todense()).ravel()

        order = np.argsort(scores)[::-1]

        keywords = []

        for i in order:

            if scores[i] <= 0:
                break

            keywords.append(names[i])

            if len(keywords) >= config.MAX_KEYWORDS:
                break

        return keywords

    except Exception as e:
        logging.exception("Keyword extraction failed")

        return []
    
def summarize_note(text: str) -> str:
    prompt = (
        get_prompt("summary")
        + f"\n\nLimit to {config.SUMMARY_MAX_WORDS} words.\n\n"
        + text
    )

    return call_llm(prompt)

def compute_embedding(text: str) -> np.ndarray:
    return embedding_model.encode(
        text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

def compute_importance(text: str) -> float:
    doc = nlp(text)

    entity_score = min(len(doc.ents) / 10, 1.0)

    length_score = min(len(text.split()) / 100, 1.0)

    keyword_score = min(len(extract_keywords(text)) / 10, 1.0)

    score = (
        entity_score +
        length_score +
        keyword_score
    ) / 3

    return round(score, 2)

def build_metadata(text: str) -> Dict[str, Any]:
    return {
        "uuid": str(uuid.uuid4()),
        "created_at": datetime.now(UTC).isoformat(),
        "characters": len(text),
        "words": len(text.split())
    }

def process_note(raw_note: str) -> ProcessedNote:

    raw_note = raw_note[:config.MAX_NOTE_LENGTH]

    clean = clean_text(raw_note)

    clean = normalize_text(clean)

    summary = summarize_note(clean)

    embedding = compute_embedding(clean)

    importance = compute_importance(clean)

    keywords = extract_keywords(clean)

    metadata = build_metadata(clean)

    return ProcessedNote(
        raw_text=raw_note,
        clean_text=clean,
        summary=summary,
        embedding=embedding,
        importance=importance,
        keywords=keywords,
        metadata=metadata,
    )

# MEMORY EXTRACTION

from dataclasses import dataclass, field

@dataclass
class EntityInfo:
    text: str
    label: str

@dataclass
class Relationship:
    source: str
    relation: str
    target: str
    confidence: float

@dataclass
class MemoryObject:

    summary: str

    topics: List[str] = field(default_factory=list)

    entities: List[EntityInfo] = field(default_factory=list)

    goals: List[str] = field(default_factory=list)

    projects: List[str] = field(default_factory=list)

    skills: List[str] = field(default_factory=list)

    interests: List[str] = field(default_factory=list)

    beliefs: List[str] = field(default_factory=list)

    observations: List[str] = field(default_factory=list)

    events: List[str] = field(default_factory=list)

    tasks: List[str] = field(default_factory=list)

    relationships: List[Relationship] = field(default_factory=list)

    metadata: Dict[str, Any] = field(default_factory=dict)

def extract_entities(text: str) -> List[EntityInfo]:

    doc = nlp(text)

    entities = []

    seen = set()

    for ent in doc.ents:

        key = (ent.text.lower(), ent.label_)

        if key in seen:
            continue

        seen.add(key)

        entities.append(
            EntityInfo(
                text=ent.text,
                label=ent.label_
            )
        )

    return entities[:config.MAX_ENTITIES]

def extract_topics(text: str) -> List[str]:

    return extract_keywords(text)

def build_memory_prompt(note: ProcessedNote) -> str:

    return f"""
{get_prompt("memory_extraction")}

Return ONLY valid JSON.

Schema:

{{
    "goals": [],
    "projects": [],
    "skills": [],
    "interests": [],
    "beliefs": [],
    "observations": [],
    "events": [],
    "tasks": [],
    "relationships": [
        {{
            "source":"",
            "relation":"",
            "target":"",
            "confidence":0.0
        }}
    ]
}}

Summary:

{note.summary}

Original Note:

{note.clean_text}
"""

MEMORY_SCHEMA = {

    "type":"object",

    "properties":{

        "goals":{"type":"array"},

        "projects":{"type":"array"},

        "skills":{"type":"array"},

        "interests":{"type":"array"},

        "beliefs":{"type":"array"},

        "observations":{"type":"array"},

        "events":{"type":"array"},

        "tasks":{"type":"array"},

        "relationships":{"type":"array"}

    }
}

def llm_extract_memory(note: ProcessedNote):

    prompt = build_memory_prompt(note)

    MAX_RETRIES = 5

    for attempt in range(MAX_RETRIES):

        try:
            text = call_llm(prompt)      # Don't pass MEMORY_SCHEMA here

            return json.loads(text)

        except json.JSONDecodeError:

            print(f"Invalid JSON. Retry {attempt+1}/{MAX_RETRIES}")

            prompt = f"""
The previous response was INVALID JSON.

Repair the JSON.

Rules:
- Output ONLY valid JSON.
- No markdown.
- No explanations.
- No comments.
- Every key must exist.
- Arrays must exist even if empty.

Previous invalid output:

{text}
"""

    raise ValueError("LLM failed to produce valid JSON after 5 attempts.")

def build_relationships(data):

    relationships = []

    for rel in data.get("relationships", []):

        try:

            relationships.append(

                Relationship(

                    source=rel["source"],

                    relation=rel["relation"],

                    target=rel["target"],

                    confidence=float(
                        rel.get(
                            "confidence",
                            1.0
                        )
                    )

                )

            )

        except Exception:

            continue

    return relationships

def extract_memory(
    note: ProcessedNote
) -> MemoryObject:

    llm = llm_extract_memory(note)

    return MemoryObject(

        summary=note.summary,

        topics=extract_topics(
            note.clean_text
        ),

        entities=extract_entities(
            note.clean_text
        ),

        goals=llm.get(
            "goals",
            []
        ),

        projects=llm.get(
            "projects",
            []
        ),

        skills=llm.get(
            "skills",
            []
        ),

        interests=llm.get(
            "interests",
            []
        ),

        beliefs=llm.get(
            "beliefs",
            []
        ),

        observations=llm.get(
            "observations",
            []
        ),

        events=llm.get(
            "events",
            []
        ),

        tasks=llm.get(
            "tasks",
            []
        ),

        relationships=build_relationships(
            llm
        ),

        metadata=note.metadata

    )

print("Extracting memory...")

# MEMORY INDEXING


@dataclass
class SearchResult:
    note_id: int
    score: float
    source: str
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class MemoryIndexData:
    faiss_index: Any = None
    graph: nx.MultiDiGraph = field(default_factory=nx.MultiDiGraph)
    timeline: Dict[str, List[int]] = field(default_factory=dict)
    keyword_lookup: Dict[str, List[int]] = field(default_factory=dict)
    embeddings: np.ndarray | None = None
    note_ids: List[int] = field(default_factory=list)


class MemoryIndex:

    def __init__(self, config):
        self.config = config
        self.data = MemoryIndexData()

    def get_connection(self):
        conn = sqlite3.connect(str(self.config.DATABASE_PATH))
        conn.row_factory = sqlite3.Row
        return conn

    def load_database(self):
        conn = self.get_connection()
        cur = conn.cursor()

        notes = [dict(r) for r in cur.execute("SELECT * FROM notes")]
        embeddings = [dict(r) for r in cur.execute("SELECT * FROM embeddings")]
        entities = [dict(r) for r in cur.execute("SELECT * FROM entities")]
        relationships = [dict(r) for r in cur.execute("SELECT * FROM relationships")]
        topics = [dict(r) for r in cur.execute("SELECT * FROM topics")]

        conn.close()

        return {
            "notes": notes,
            "embeddings": embeddings,
            "entities": entities,
            "relationships": relationships,
            "topics": topics,
        }

    def build_vector_index(self, db):
        vectors = []
        ids = []

        for row in db["embeddings"]:
            vec = np.frombuffer(row["vector"], dtype=np.float32)
            vectors.append(vec)
            ids.append(row["note_id"])

        if not vectors:
            return

        vectors = np.vstack(vectors).astype(np.float32)
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)

        self.data.faiss_index = index
        self.data.embeddings = vectors
        self.data.note_ids = ids

    def build_graph(self, db):
        g = nx.MultiDiGraph()

        for e in db["entities"]:
            g.add_node(e["entity"], label=e["label"])

        for r in db["relationships"]:
            g.add_edge(
                r["source_entity"],
                r["target_entity"],
                relation=r["relationship"],
                confidence=r["confidence"],
            )

        self.data.graph = g

    def build_timeline(self, db):
        timeline = {}

        for note in db["notes"]:
            date = note["created_at"][:7]
            timeline.setdefault(date, []).append(note["id"])

        self.data.timeline = timeline

    def build_keyword_index(self, db):
        lookup = {}

        for topic in db["topics"]:
            lookup.setdefault(topic["topic"].lower(), []).append(topic["note_id"])

        self.data.keyword_lookup = lookup

    def build_memory_index(self):
        db = self.load_database()
        self.build_vector_index(db)
        self.build_graph(db)
        self.build_timeline(db)
        self.build_keyword_index(db)

        if self.data.faiss_index is not None:
         faiss.write_index(
        self.data.faiss_index,
        str(self.config.FAISS_INDEX_PATH),
    )

        with open(self.config.GRAPH_PATH, "wb") as f:
            pickle.dump(self.data.graph, f)

    def search_vector(self, query_embedding, top_k=None):
        if top_k is None:
            top_k = self.config.TOP_K_VECTOR_RESULTS

        scores, ids = self.data.faiss_index.search(
            query_embedding.astype(np.float32),
            top_k,
        )

        return list(zip(scores[0], ids[0]))

    def search_keyword(self, keyword):
        return self.data.keyword_lookup.get(keyword.lower(), [])

    def search_graph(self, entity):
        if entity not in self.data.graph:
            return []
        return list(self.data.graph.neighbors(entity))

    def search_timeline(self, month):
        return self.data.timeline.get(month, [])

    def search_memory(self, query_embedding, keyword=None, entity=None, month=None):
        return {
            "vector": self.search_vector(query_embedding),
            "keyword": self.search_keyword(keyword) if keyword else [],
            "graph": self.search_graph(entity) if entity else [],
            "timeline": self.search_timeline(month) if month else [],
        }
    


# MEMORY INTELLIGENCE
@dataclass
class ConsolidatedMemory:
    title: str
    description: str
    source_ids: List[int] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class AnalysisResult:
    profile: Dict[str, Any] = field(default_factory=dict)
    skills: List[str] = field(default_factory=list)
    interests: List[str] = field(default_factory=list)
    goals: List[str] = field(default_factory=list)
    habits: List[str] = field(default_factory=list)
    contradictions: List[str] = field(default_factory=list)
    predictions: List[str] = field(default_factory=list)


class MemoryIntelligence:

    def __init__(self, llm_client=None, data_layer=None):
        self.llm_client = llm_client
        self.data_layer = data_layer

    def load_memories(self):
        return load_records("memories")

    def group_related_memories(self, memories):
        groups = {}

        for memory in memories:
            key = memory.get("memory_type", "general")
            groups.setdefault(key, []).append(memory)

        return groups

    def consolidate_memories(self):

        memories = self.load_memories()

        groups = self.group_related_memories(memories)

        consolidated = []

        for category, items in groups.items():

            prompt = (
                "Merge the following related memories into one higher-level memory:\n\n"
                + "\n".join(item["content"] for item in items)
            )

            summary = call_llm(prompt)

            consolidated.append(
                ConsolidatedMemory(
                    title=category,
                    description=summary,
                    source_ids=[item["id"] for item in items],
                )
            )

        return consolidated

    def analyze_memory(self, consolidated):

        prompt = (
            "Analyze the following consolidated memories and identify "
            "skills, interests, goals, habits, contradictions and predictions.\n\n"
            + "\n\n".join(memory.description for memory in consolidated)
        )

        result = call_llm(prompt)

        return AnalysisResult(
            profile={
                "raw": result
            }
        )

    def save_analysis(self, analysis):

        save_record(
            "metadata",
            {
                "note_id": None,
                "key": "analysis",
                "value": str(analysis.profile),
            },
        )

    def run(self):

        consolidated = self.consolidate_memories()

        analysis = self.analyze_memory(consolidated)

        self.save_analysis(analysis)

        return analysis


# CONVERSATIONAL ENGINE

@dataclass
class ChatMessage:
    role: str
    content: str


class ConversationalEngine:

    def __init__(self, memory_index, llm_client, embedding_model):
        self.memory_index = memory_index
        self.llm_client = llm_client
        self.embedding_model = embedding_model
        self.history: List[ChatMessage] = []

    def embed_query(self, query: str):
        return self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    def retrieve_context(self, query: str):
        embedding = self.embed_query(query)

        return self.memory_index.search_memory(
            query_embedding=embedding,
            keyword=query,
        )

    def build_prompt(self, query: str, context: Dict[str, Any]) -> str:

        return f"""
{{
Question:
{query}

Retrieved Memory:
{context}

Instructions:
- Answer ONLY from retrieved memories.
- If information is missing, explicitly state it.
- Do not invent facts.
}}
"""

    def answer_question(self, query: str):

        context = self.retrieve_context(query)

        prompt = self.build_prompt(query, context)

        answer = call_llm(prompt)

        self.history.append(ChatMessage("user", query))
        self.history.append(ChatMessage("assistant", answer))

        return answer

    def interactive_chat(self):

        print("Memory AI Chat (type 'exit' to quit)")

        while True:

            query = input("> ")

            if query.lower() in {"exit", "quit"}:
                break

            print(self.answer_question(query))

    def clear_history(self):
        self.history.clear()

    def export_history(self):

        return [
            {
                "role": msg.role,
                "content": msg.content
            }
            for msg in self.history
        ]

from pathlib import Path
from pathlib import Path


class MemoryPipeline:

    def __init__(
        self,
        data_layer,
        note_processing,
        memory_extraction,
        memory_index,
        memory_intelligence,
        conversation_engine,
    ):

        self.db = data_layer
        self.note_processing = note_processing
        self.memory_extraction = memory_extraction
        self.memory_index = memory_index
        self.memory_intelligence = memory_intelligence
        self.chat = conversation_engine

    def import_notes(self, folder):

        folder = Path(folder)

        notes = []

        for file in folder.glob("*.txt"):
            notes.append(
                (
                    file.name,
                    file.read_text(encoding="utf-8"),
                )
            )

        return notes

    def process_and_store(self, raw_text):

        note = process_note(raw_text)

        memory = extract_memory(note)

        note_id = save_record(
            "notes",
            {
                "uuid": note.metadata["uuid"],
                "raw_text": note.raw_text,
                "clean_text": note.clean_text,
                "summary": note.summary,
                "importance": note.importance,
                "created_at": note.metadata["created_at"],
            },
        )

        save_record(
            "embeddings",
            {
                "note_id": note_id,
                "vector": note.embedding.astype(np.float32).tobytes(),
                "dimension": len(note.embedding),
            },
        )

        for topic in memory.topics:
            save_record(
                "topics",
                {
                    "note_id": note_id,
                    "topic": topic,
                    "score": 1.0,
                },
            )

        for entity in memory.entities:
            save_record(
                "entities",
                {
                    "note_id": note_id,
                    "entity": entity.text,
                    "label": entity.label,
                },
            )

        for rel in memory.relationships:
            save_record(
                "relationships",
                {
                    "source_entity": rel.source,
                    "relationship": rel.relation,
                    "target_entity": rel.target,
                    "confidence": rel.confidence,
                },
            )

        categories = {
            "goals": memory.goals,
            "projects": memory.projects,
            "skills": memory.skills,
            "interests": memory.interests,
            "beliefs": memory.beliefs,
            "observations": memory.observations,
            "events": memory.events,
            "tasks": memory.tasks,
        }

        for memory_type, items in categories.items():
            for item in items:
                save_record(
                    "memories",
                    {
                        "note_id": note_id,
                        "memory_type": memory_type,
                        "content": (
    json.dumps(item)
    if isinstance(item, (dict, list))
    else str(item)
),
                        "confidence": 1.0,
                    },
                )

        return note, memory
    
    

    def build_indexes(self):

        self.memory_index.build_memory_index()

    def consolidate_and_analyze(self):

        consolidated = self.memory_intelligence.consolidate_memories()

        return self.memory_intelligence.analyze_memory(consolidated)

    def answer(self, query):

        return self.chat.answer_question(query)

    def run_pipeline(self, notes_folder):

        initialize_database()

        notes = self.import_notes(notes_folder)

        from concurrent.futures import ThreadPoolExecutor

        with ThreadPoolExecutor(max_workers=8) as executor:
         
         list(executor.map(
        lambda x: self.process_and_store(x[1]),
        notes
    ))

        self.build_indexes()

        self.consolidate_and_analyze()

        print("Pipeline completed successfully.")




memory_index = MemoryIndex(config)

memory_intelligence = MemoryIntelligence()

chat_engine = ConversationalEngine(
    memory_index=memory_index,
    llm_client=client,
    embedding_model=embedding_model,
)

pipeline = MemoryPipeline(
    data_layer=None,
    note_processing=None,
    memory_extraction=None,
    memory_index=memory_index,
    memory_intelligence=memory_intelligence,
    conversation_engine=chat_engine,
)
pipeline.run_pipeline("notes")